# Demo Cohort Check — MIMIC-IV Clinical Database Demo (v2.2)

Loads `data/processed/cohort_features.parquet` (built by `src/build_demo_cohort.py` from the small, public MIMIC-IV demo) and summarizes: N, deaths, per-feature summary statistics, and missingness.

This notebook only reads the already-built parquet file — it does not touch the raw demo CSVs directly. Run `src/build_demo_cohort.py` first (see the command below).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger
from src.build_demo_cohort import SELECTED_NUMERIC_FEATURES

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])

TARGET_COL = config["demo_cohort"]["target_column"]
COHORT_PATH = config["demo_cohort"]["output_path"]

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
from pathlib import Path

if not Path(COHORT_PATH).exists():
    raise FileNotFoundError(
        f"'{COHORT_PATH}' not found. Build it first:\n\n"
        "    python -m src.build_demo_cohort\n\n"
        "(after extracting the MIMIC-IV Clinical Database Demo v2.2 into "
        f"{config['demo_cohort']['demo_data_dir']})"
    )

df = pd.read_parquet(COHORT_PATH)
print(f"Loaded {COHORT_PATH}: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Cohort size and outcome (N, deaths)

In [ ]:
n_patients = len(df)
n_deaths = int(df[TARGET_COL].sum())
death_rate = df[TARGET_COL].mean()

print(f"N patients: {n_patients}")
print(f"N deaths (in-hospital): {n_deaths}")
print(f"Death rate: {death_rate:.1%}")

if n_patients < 50:
    print(
        "\nNOTE: the MIMIC-IV demo has ~100 patients total, and this cohort "
        "applies further inclusion criteria (first ICU stay, >=24h LOS, "
        "age>=18), so N here will be small. Treat any downstream modeling "
        "on this demo cohort as a pipeline smoke test, not a clinically "
        "meaningful result — sample size is far too small for that."
    )

## 2. Feature summary (numeric features actually available in this extract)

In [ ]:
available_features = [c for c in SELECTED_NUMERIC_FEATURES if c in df.columns]
missing_from_extract = [c for c in SELECTED_NUMERIC_FEATURES if c not in df.columns]

print(f"Numeric features available ({len(available_features)}/{len(SELECTED_NUMERIC_FEATURES)}): {available_features}")
if missing_from_extract:
    print(f"Requested but NOT buildable from this demo extract: {missing_from_extract}")

df[available_features].describe().T

## 3. Missingness per feature

In [ ]:
missingness = (
    df[available_features].isna().sum().rename("n_missing").to_frame()
)
missingness["pct_missing"] = 100 * missingness["n_missing"] / len(df)
missingness.sort_values("pct_missing", ascending=False)

## 4. Demographics and outcome by group (quick sanity check)

In [ ]:
if "gender" in df.columns:
    print("By gender:")
    print(df.groupby("gender")[TARGET_COL].agg(["count", "sum", "mean"]))

if "first_careunit" in df.columns:
    print("\nBy first ICU careunit:")
    print(df.groupby("first_careunit")[TARGET_COL].agg(["count", "sum", "mean"]))

## Summary

> **Target column name mismatch:** this cohort's outcome column is named `"mortality"`. The rest of this project's pipeline (`src/preprocess.py` onward) expects `configs/config.yaml -> preprocessing.target_column`, currently `"in_hospital_mortality"`. Before running `notebooks/01_cohort_audit.ipynb` or later against this demo cohort, either update that config value to `"mortality"`, or rename the column.
>
> **Sample size:** the MIMIC-IV demo is ~100 patients before any inclusion criteria — this is sufficient to smoke-test the pipeline end-to-end, but far too small to draw any clinical conclusion from.